In [1]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# 定数
DROPDOWN_DESCRIPTION_LIST = [ '比較元', '比較先' ]
DROPDOWN_VALUE_INDEX_LIST = [ 0, -1 ]
CHECKBOX_DESCRIPTION_LIST = [ f'{desc}を詳細化' for desc in DROPDOWN_DESCRIPTION_LIST ]
CHECKBOX_DESC_TO_INDEX_DICT = { desc:index for index,desc in enumerate(CHECKBOX_DESCRIPTION_LIST) }
DEFAULT_OPTION_TYPE = 'simple'
options_dict = {
    'simple': [('exp00_01(score=0.505)', 'exp00_01'), ('exp01_10(score=0.610)', 'exp01_10'), ('exp03_05(score=0.725)', 'exp03_05')],
    'detail': [('exp00_01(score=0.505)', 'exp00_01'), ('exp01_10(score=0.610)', 'exp01_10'), ('exp02_08(score=0.580)', 'exp01_10'), ('exp03_05(score=0.725)', 'exp03_05')],
}

# UI部品
upload_widget = widgets.FileUpload(
    description='結果xlsxを選択',
    accept='.xlsx',
    multiple=False,
    button_style='info',
)

filename_area = widgets.Output()
with filename_area:
    print("ここに結果ファイル名を表示します")

exec_button = widgets.Button(
    description='解析実行',
    button_style='primary',
)

dropdown = [
    widgets.Dropdown(
        options=options_dict[DEFAULT_OPTION_TYPE],
        description=desc,
        value=options_dict[DEFAULT_OPTION_TYPE][value_index][1],
    ) for desc,value_index in zip(DROPDOWN_DESCRIPTION_LIST, DROPDOWN_VALUE_INDEX_LIST)
]

checkbox = [
    widgets.Checkbox(
        value=False,
        description=desc,
    ) for desc in CHECKBOX_DESCRIPTION_LIST
]

hbox = widgets.HBox(
    [ exec_button ] +
    [
        widgets.VBox([
            dropdown[index],
            checkbox[index],
        ]) for index,_ in enumerate(dropdown)
    ]
)

output_area = widgets.Output()
with output_area:
    print("ここに解析結果を表示します")

vbox = widgets.VBox([
    upload_widget,
    filename_area,
    hbox,
    output_area,
])
display(vbox)

# イベントハンドラ
def on_checkbox_change(change):
    checkbox_desc = change['owner'].description
    index = CHECKBOX_DESC_TO_INDEX_DICT[checkbox_desc]
    dropdown[index].options = options_dict['detail'] if change['new'] else options_dict['simple']
    dropdown[index].value = dropdown[index].options[DROPDOWN_VALUE_INDEX_LIST[index]][1]
for index,_ in enumerate(checkbox):
    checkbox[index].observe(on_checkbox_change, names='value')

def on_upload_change(change):
    if upload_widget.value:
        uploaded_file = upload_widget.value[0]
        file_name = uploaded_file.name
        file_content = uploaded_file.content
        with filename_area:
            clear_output()
            print(f"ファイル名: {file_name}")
upload_widget.observe(on_upload_change, names='value')

def on_button_click(b):
    # 解析実行
    params = [ dropdown[index].value for index,_ in enumerate(dropdown) ]
    with output_area:
        clear_output()
        # 結果表示
        #print(f"{' + '.join([ str(x) for x in params ])} = {sum(params)}")
        print(f"{' + '.join([ str(x) for x in params ])}")
exec_button.on_click(on_button_click)